# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neel0289/FlyRank-Week1A1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:
Prioritize pages with meaningful search demand and a CTR that is lower than expected for their average search position. Pages with a larger CTR-position gap receive higher priority.

Reason code:
CTR_POSITION_GAP — the page has a meaningful CTR underperformance relative to its search position and enough search demand to justify review.

Action:
CTR_FIX — review the page title, meta description, search intent alignment, and other factors that may improve click-through rate.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load the dataset
# --------------------------------------------------

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))


# --------------------------------------------------
# 2. Keep rows with the signals needed by our rule
# --------------------------------------------------

df = df.dropna(
    subset=["ctr", "avg_position", "search_volume"]
).copy()


# --------------------------------------------------
# 3. Create position buckets
# --------------------------------------------------

position_bins = [0, 1, 3, 5, 10, 20, np.inf]

position_labels = [
    "1",
    "2-3",
    "4-5",
    "6-10",
    "11-20",
    "21+"
]

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)


# --------------------------------------------------
# 4. Calculate expected CTR for each position bucket
# --------------------------------------------------

expected_ctr = (
    df.groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .median()
)

df["expected_ctr"] = (
    df["position_bucket"]
    .map(expected_ctr)
)


# --------------------------------------------------
# 5. Calculate CTR-position gap
# --------------------------------------------------

df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr"]
).clip(lower=0)


# --------------------------------------------------
# 6. Normalize CTR gap
# --------------------------------------------------

ctr_max = df["ctr_gap"].max()

if ctr_max > 0:
    df["ctr_gap_score"] = (
        df["ctr_gap"] / ctr_max
    )
else:
    df["ctr_gap_score"] = 0


# --------------------------------------------------
# 7. Normalize search volume
# --------------------------------------------------

df["volume_log"] = np.log1p(
    df["search_volume"]
)

volume_max = df["volume_log"].max()

if volume_max > 0:
    df["volume_score"] = (
        df["volume_log"] / volume_max
    )
else:
    df["volume_score"] = 0


# --------------------------------------------------
# 8. Create the baseline score
# --------------------------------------------------

df["score"] = (
    0.7 * df["ctr_gap_score"]
    + 0.3 * df["volume_score"]
)


# --------------------------------------------------
# 9. Add reason code and action
# --------------------------------------------------

df["reason_code"] = "CTR_POSITION_GAP"

df["action"] = "CTR_FIX"


# --------------------------------------------------
# 10. Create ranked queue
# --------------------------------------------------

queue = (
    df[
        [
            "content_id",
            "score",
            "reason_code",
            "action",
            "ctr",
            "avg_position",
            "search_volume",
            "expected_ctr",
            "ctr_gap"
        ]
    ]
    .sort_values(
        "score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "rank",
    queue.index + 1
)


# --------------------------------------------------
# 11. Write the required CSV
# --------------------------------------------------

output_path = Path(
    "../outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    output_path,
    index=False
)


# --------------------------------------------------
# 12. Display results
# --------------------------------------------------

print("\nRanked queue created successfully.")
print("Rows ranked:", len(queue))
print("CSV:", output_path)

print("\nTop 20:")
display(queue.head(20))

Rows: 30000
Columns: 44

Ranked queue created successfully.
Rows ranked: 27532
CSV: ..\outputs\baseline_action_score.csv

Top 20:


,rank,content_id,score,reason_code,action,ctr,avg_position,search_volume,expected_ctr,ctr_gap
0,1,content_fecb3b1efda1,0.919116,CTR_POSITION_GAP,CTR_FIX,0.0,3.5,3600.0,0.24,0.24
1,2,content_da3f2c1bef3f,0.913332,CTR_POSITION_GAP,CTR_FIX,0.0,4.1,2900.0,0.24,0.24
2,3,content_31b10f132c97,0.913332,CTR_POSITION_GAP,CTR_FIX,0.0,3.3,2900.0,0.24,0.24
3,4,content_3474a43ad37f,0.908270,CTR_POSITION_GAP,CTR_FIX,0.0,4.0,2400.0,0.24,0.24
4,5,content_5d8cc9246c52,0.902022,CTR_POSITION_GAP,CTR_FIX,0.0,3.3,1900.0,0.24,0.24
5,6,content_d56f7f74c15c,0.902022,CTR_POSITION_GAP,CTR_FIX,0.0,3.6,1900.0,0.24,0.24
6,7,content_991b77f0d406,0.897427,CTR_POSITION_GAP,CTR_FIX,0.0,3.4,1600.0,0.24,0.24
7,8,content_5b5e85993c2b,0.891875,CTR_POSITION_GAP,CTR_FIX,0.0,4.3,1300.0,0.24,0.24
8,9,content_64fdf67a3b9e,0.881444,CTR_POSITION_GAP,CTR_FIX,0.0,5.0,880.0,0.24,0.24
9,10,content_cfa62c0036fa,0.881444,CTR_POSITION_GAP,CTR_FIX,0.0,4.0,880.0,0.24,0.24


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Get the top 20 ranked pages
top20 = queue.head(20).copy()

# Add a confidence note based on the strength of the CTR gap
top20["confidence_note"] = np.where(
    top20["ctr_gap"] >= top20["ctr_gap"].quantile(0.75),
    "High confidence: large CTR gap for the page's search position.",
    np.where(
        top20["ctr_gap"] >= top20["ctr_gap"].quantile(0.50),
        "Medium confidence: meaningful CTR gap, but the opportunity is less strong.",
        "Lower confidence: ranking is more dependent on search demand than CTR gap."
    )
)

# Add a skeptical explanation for each recommendation
top20["what_would_make_it_wrong"] = (
    "The low CTR may be appropriate for the page's search intent, "
    "or changing the page metadata may not increase clicks."
)

# Display the review
review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "score",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,content_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_fecb3b1efda1,CTR_FIX,CTR_POSITION_GAP,0.919116,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
1,2,content_da3f2c1bef3f,CTR_FIX,CTR_POSITION_GAP,0.913332,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
2,3,content_31b10f132c97,CTR_FIX,CTR_POSITION_GAP,0.913332,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
3,4,content_3474a43ad37f,CTR_FIX,CTR_POSITION_GAP,0.908270,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
4,5,content_5d8cc9246c52,CTR_FIX,CTR_POSITION_GAP,0.902022,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
5,6,content_d56f7f74c15c,CTR_FIX,CTR_POSITION_GAP,0.902022,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
6,7,content_991b77f0d406,CTR_FIX,CTR_POSITION_GAP,0.897427,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
7,8,content_5b5e85993c2b,CTR_FIX,CTR_POSITION_GAP,0.891875,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
8,9,content_64fdf67a3b9e,CTR_FIX,CTR_POSITION_GAP,0.881444,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...
9,10,content_cfa62c0036fa,CTR_FIX,CTR_POSITION_GAP,0.881444,High confidence: large CTR gap for the page's ...,The low CTR may be appropriate for the page's ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# 1. Find potentially weak picks
# --------------------------------------------------

# Weak picks are high-ranked pages where the CTR gap is small
# but the page still received a relatively high score.

weak_picks = (
    queue
    .sort_values("score", ascending=False)
    .query("ctr_gap <= @queue['ctr_gap'].median()")
    .head(5)
)

print("Potentially weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "score",
            "action",
            "reason_code",
            "ctr",
            "avg_position",
            "search_volume",
            "ctr_gap"
        ]
    ]
)


# --------------------------------------------------
# 2. Leakage check
# --------------------------------------------------

# Columns that should NOT be used as inputs to this baseline
forbidden_terms = [
    "opportunity",
    "label",
    "target",
    "future",
    "june",
    "flag"
]

used_columns = [
    "ctr",
    "avg_position",
    "search_volume"
]

leaked_columns = [
    col for col in used_columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("\nLeakage check:")

if leaked_columns:
    print("WARNING: Potential leakage columns found:", leaked_columns)
else:
    print("PASS: No target, future-window, or product-flag columns are used.")

print("\nBaseline input columns:")
print(used_columns)

Potentially weak picks:


,rank,content_id,score,action,reason_code,ctr,avg_position,search_volume,ctr_gap
7034,7035,content_ef99c4abd9ab,0.300000,CTR_FIX,CTR_POSITION_GAP,0.03,38.5,74000.0,0.0
7117,7118,content_bf67a444faef,0.294611,CTR_FIX,CTR_POSITION_GAP,0.00,45.5,60500.0,0.0
7118,7119,content_454cc6654c6e,0.294611,CTR_FIX,CTR_POSITION_GAP,0.00,44.9,60500.0,0.0
7119,7120,content_5ec29ae79c60,0.294611,CTR_FIX,CTR_POSITION_GAP,0.00,49.8,60500.0,0.0
7120,7121,content_deb54e9e19cd,0.294611,CTR_FIX,CTR_POSITION_GAP,0.00,41.7,60500.0,0.0



Leakage check:
PASS: No target, future-window, or product-flag columns are used.

Baseline input columns:
['ctr', 'avg_position', 'search_volume']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.